# 05_xray_console — State x-ray via OpenTelemetry (console output)

`OpenTelemetryPlugin` wires into `ActionMachine` as a plugin.
**No changes to Action code** — all observability is in the machine setup.

Two optional signals, both can be combined:

| Signal | Provider | What you get |
|--------|----------|-------------|
| **Traces** | `tracer_provider` | Root span + child span per aspect; timings, status |
| **Logs (x-ray)** | `logger_provider` | Log record per lifecycle event; `aoa.state.*` attributes after each step |

This notebook uses `ConsoleSpanExporter` and `ConsoleLogRecordExporter`
so you can see everything in the cell output without any backend.

In [ ]:
!pip install -q aoa-action-machine aoa-otel

In [2]:
import asyncio

from opentelemetry.sdk._logs import LoggerProvider
from opentelemetry.sdk._logs.export import ConsoleLogRecordExporter, SimpleLogRecordProcessor
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor
from pydantic import Field

from aoa.action_machine.auth import GuestRole
from aoa.action_machine.context import Context
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.aspects import regular_aspect, summary_aspect
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.checkers import result_float, result_int, result_string
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.model import BaseAction, BaseParams, BaseResult
from aoa.action_machine.resources import Connections
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine
from aoa.otel import OpenTelemetryPlugin

## Domain, Params, Result

In [3]:
class CheckoutDomain(BaseDomain):
    name = "checkout"
    description = "E-commerce checkout pipeline"


class CheckoutParams(BaseParams):
    sku: str = Field(description="Product SKU")
    quantity: int = Field(description="Number of units")
    unit_price: float = Field(description="Price per unit in USD")


class CheckoutResult(BaseResult):
    sku: str = Field(description="Validated SKU")
    total: float = Field(description="Order total in USD")
    confirmation: str = Field(description="Human-readable confirmation message")

## Action — three steps, each with checked state

After `validate_aspect`: `state = {"sku": ..., "quantity": ...}`
After `enrich_aspect`: `state = {"sku": ..., "quantity": ..., "total": ...}`

The x-ray log records will show exactly these fields at each step.

In [4]:
from typing import Any

from aoa.action_machine.model.base_state import BaseState
from aoa.action_machine.resources.base_resource import BaseResource
from aoa.action_machine.runtime.tools_box import ToolsBox


@meta(description="Three-step checkout: validate → enrich → confirm", domain=CheckoutDomain)
@check_roles(GuestRole)
class CheckoutAction(BaseAction[CheckoutParams, CheckoutResult]):

    @regular_aspect("Step 1: Validate SKU and quantity")
    @result_string("sku", required=True, min_length=3)
    @result_int("quantity", required=True, min_value=1, max_value=500)
    async def validate_aspect(self, params: CheckoutParams, state: BaseState, box: ToolsBox, connections: Connections) -> dict[str, Any]:
        return {"sku": params.sku.strip().upper(), "quantity": params.quantity}

    @regular_aspect("Step 2: Calculate total")
    @result_string("sku", required=True)
    @result_int("quantity", required=True)
    @result_float("total", required=True, min_value=0.01)
    async def enrich_aspect(self, params: CheckoutParams, state: BaseState, box: ToolsBox, connections: Connections) -> dict[str, Any]:
        return {
            "sku": state["sku"],
            "quantity": state["quantity"],
            "total": round(state["quantity"] * params.unit_price, 2),
        }

    @summary_aspect("Step 3: Confirm order")
    async def confirm_summary(self, params: CheckoutParams, state: BaseState, box: ToolsBox, connections: Connections) -> CheckoutResult:
        return CheckoutResult(
            sku=state["sku"],
            total=state["total"],
            confirmation=f"Order confirmed: {state['quantity']}x {state['sku']} = ${state['total']}",
        )

## OTel setup — console exporters

Both providers are optional. Using both together enables automatic log-to-trace
correlation: every log record carries the same `trace_id` as the root span,
and after-step logs carry the `span_id` of the corresponding child span.

In [5]:
tp = TracerProvider()
tp.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))

lp = LoggerProvider()
lp.add_log_record_processor(SimpleLogRecordProcessor(ConsoleLogRecordExporter()))

plugin = OpenTelemetryPlugin(
    tracer_provider=tp,
    logger_provider=lp,
    service_name="checkout-service",
)

## Run

Watch the output for two types of JSON objects:
- Spans: `"name": "validate_aspect"` — timing and structure
- Logs: `"body": "aoa.aspect.regular.after"` — state x-ray with `aoa.state.*` attributes

After `validate_aspect` you'll see `aoa.state.sku` and `aoa.state.quantity`.
After `enrich_aspect` you'll see those two plus `aoa.state.total`.
That's the x-ray: each record shows the exact state after that step.

In [6]:
machine = ActionProductMachine(plugins=[plugin])
result = await machine.run(
    Context(),
    CheckoutAction(),
    CheckoutParams(sku=" widget-42 ", quantity=3, unit_price=19.99),
)
print(f"\nFinal result: {result.confirmation}")

{
    "body": "aoa.action.start",
    "severity_number": 9,
    "severity_text": "INFO",
    "attributes": {
        "aoa.action": "CheckoutAction",
        "aoa.nest_level": 1
    },
    "dropped_attributes": 0,
    "timestamp": "2026-11-08T18:10:05.690380Z",
    "observed_timestamp": "2026-11-08T18:10:05.690380Z",
    "trace_id": "0x12e8d064f3cc91f6726876cb055ea1cd",
    "span_id": "0x3f8ab9813230fab6",
    "trace_flags": 3,
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.45.1",
            "service.instance.id": "bd0d3c13-5864-4f9e-ade0-5d41b4471a5b",
            "service.name": "unknown_service:python3"
        },
        "schema_url": ""
    },
    "event_name": ""
}


{
    "body": "aoa.aspect.regular.before",
    "severity_number": 9,
    "severity_text": "INFO",
    "attributes": {
        "aoa.action": "CheckoutAction",
        "aoa.aspect": "validate_aspect"
    },
    "dropped_attributes": 0,
    "timestamp": "2026-11-08T18:10:05.691157Z",
    "observed_timestamp": "2026-11-08T18:10:05.691157Z",
    "trace_id": "0x12e8d064f3cc91f6726876cb055ea1cd",
    "span_id": "0x3f8ab9813230fab6",
    "trace_flags": 3,
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.45.1",
            "service.instance.id": "bd0d3c13-5864-4f9e-ade0-5d41b4471a5b",
            "service.name": "unknown_service:python3"
        },
        "schema_url": ""
    },
    "event_name": ""
}


{
    "name": "validate_aspect",
    "context": {
        "trace_id": "0x12e8d064f3cc91f6726876cb055ea1cd",
        "span_id": "0xeab1447dd7f1b3be",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x3f8ab9813230fab6",
    "start_time": "2026-11-08T18:10:05.691152Z",
    "end_time": "2026-11-08T18:10:05.691646Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "aoa.aspect": "validate_aspect",
        "aoa.action": "CheckoutAction",
        "aoa.duration_ms": 0.04887580871582031
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.45.1",
            "service.instance.id": "bd0d3c13-5864-4f9e-ade0-5d41b4471a5b",
            "service.name": "unknown_service:python3"
        },
        "schema_url": ""
    }
}


{
    "body": "aoa.aspect.regular.after",
    "severity_number": 9,
    "severity_text": "INFO",
    "attributes": {
        "aoa.action": "CheckoutAction",
        "aoa.aspect": "validate_aspect",
        "aoa.duration_ms": 0.04887580871582031,
        "aoa.state.sku": "WIDGET-42",
        "aoa.state.quantity": 3
    },
    "dropped_attributes": 0,
    "timestamp": "2026-11-08T18:10:05.691899Z",
    "observed_timestamp": "2026-11-08T18:10:05.691899Z",
    "trace_id": "0x12e8d064f3cc91f6726876cb055ea1cd",
    "span_id": "0xeab1447dd7f1b3be",
    "trace_flags": 3,
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.45.1",
            "service.instance.id": "bd0d3c13-5864-4f9e-ade0-5d41b4471a5b",
            "service.name": "unknown_service:python3"
        },
        "schema_url": ""
    },
    "event_name": ""
}


{
    "body": "aoa.aspect.regular.before",
    "severity_number": 9,
    "severity_text": "INFO",
    "attributes": {
        "aoa.action": "CheckoutAction",
        "aoa.aspect": "enrich_aspect"
    },
    "dropped_attributes": 0,
    "timestamp": "2026-11-08T18:10:05.692183Z",
    "observed_timestamp": "2026-11-08T18:10:05.692183Z",
    "trace_id": "0x12e8d064f3cc91f6726876cb055ea1cd",
    "span_id": "0x3f8ab9813230fab6",
    "trace_flags": 3,
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.45.1",
            "service.instance.id": "bd0d3c13-5864-4f9e-ade0-5d41b4471a5b",
            "service.name": "unknown_service:python3"
        },
        "schema_url": ""
    },
    "event_name": ""
}


{
    "name": "enrich_aspect",
    "context": {
        "trace_id": "0x12e8d064f3cc91f6726876cb055ea1cd",
        "span_id": "0x25bf6862497a7746",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x3f8ab9813230fab6",
    "start_time": "2026-11-08T18:10:05.692180Z",
    "end_time": "2026-11-08T18:10:05.692447Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "aoa.aspect": "enrich_aspect",
        "aoa.action": "CheckoutAction",
        "aoa.duration_ms": 0.031948089599609375
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.45.1",
            "service.instance.id": "bd0d3c13-5864-4f9e-ade0-5d41b4471a5b",
            "service.name": "unknown_service:python3"
        },
        "schema_url": ""
    }
}


... (3 outputs left out)

{
    "body": "aoa.aspect.summary.after",
    "severity_number": 9,
    "severity_text": "INFO",
    "attributes": {
        "aoa.action": "CheckoutAction",
        "aoa.aspect": "confirm_summary",
        "aoa.duration_ms": 0.04291534423828125,
        "aoa.result.sku": "WIDGET-42",
        "aoa.result.total": 59.97,
        "aoa.result.confirmation": "Order confirmed: 3x WIDGET-42 = $59.97"
    },
    "dropped_attributes": 0,
    "timestamp": "2026-11-08T18:10:05.693417Z",
    "observed_timestamp": "2026-11-08T18:10:05.693417Z",
    "trace_id": "0x12e8d064f3cc91f6726876cb055ea1cd",
    "span_id": "0xf0f1339ecd6397b8",
    "trace_flags": 3,
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.45.1",
            "service.instance.id": "bd0d3c13-5864-4f9e-ade0-5d41b4471a5b",
            "service.name": "unknown_service:python3"
        },
        "schema_url": 

{
    "name": "CheckoutAction",
    "context": {
        "trace_id": "0x12e8d064f3cc91f6726876cb055ea1cd",
        "span_id": "0x3f8ab9813230fab6",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-11-08T18:10:05.690369Z",
    "end_time": "2026-11-08T18:10:05.693677Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "aoa.action": "CheckoutAction",
        "aoa.nest_level": 1,
        "aoa.duration_ms": 3.606081008911133
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.45.1",
            "service.instance.id": "bd0d3c13-5864-4f9e-ade0-5d41b4471a5b",
            "service.name": "unknown_service:python3"
        },
        "schema_url": ""
    }
}


{
    "body": "aoa.action.finish",
    "severity_number": 9,
    "severity_text": "INFO",
    "attributes": {
        "aoa.action": "CheckoutAction",
        "aoa.duration_ms": 3.606081008911133,
        "aoa.result.sku": "WIDGET-42",
        "aoa.result.total": 59.97,
        "aoa.result.confirmation": "Order confirmed: 3x WIDGET-42 = $59.97"
    },
    "dropped_attributes": 0,
    "timestamp": "2026-11-08T18:10:05.693912Z",
    "observed_timestamp": "2026-11-08T18:10:05.693912Z",
    "trace_id": "0x12e8d064f3cc91f6726876cb055ea1cd",
    "span_id": "0x3f8ab9813230fab6",
    "trace_flags": 3,
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.45.1",
            "service.instance.id": "bd0d3c13-5864-4f9e-ade0-5d41b4471a5b",
            "service.name": "unknown_service:python3"
        },
        "schema_url": ""
    },
    "event_name": ""
}



Final result: Order confirmed: 3x WIDGET-42 = $59.97
